# A little extra!

## New addition to Week 1

### The Unreasonable Effectiveness of the Agent Loop

# What is an Agent?

## Three competing definitions

1. AI systems that can do work for you independently - Sam Altman

2. A system in which an LLM controls the workflow - Anthropic

3. An LLM agent runs tools in a loop to achieve a goal

## The third one is the new, emerging definition

But what does it mean?

Let's make it real.

In [1]:
# Start with some imports - rich is a library for making formatted text output in the terminal

from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
import os
load_dotenv(override=True)

True

In [2]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [3]:
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
openai = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=OPENROUTER_API_KEY)

In [4]:
# Some lists!
todos = []
completed = []

In [5]:
def get_todo_report() -> str:
    result = ""
    for index, todo in enumerate(todos):
        if completed[index]:
            result += f"Todo #{index + 1}: [green][strike]{todo}[/strike][/green]\n"
        else:
            result += f"Todo #{index + 1}: {todo}\n"
    show(result)
    return result

In [6]:
get_todo_report()

''

In [7]:
def create_todos(descriptions: list[str]) -> str:
    todos.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_todo_report()

In [8]:
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(todos):
        completed[index - 1] = True
    else:
        return "No todo at this index."
    Console().print(completion_notes)
    return get_todo_report()

In [9]:
todos, completed = [], []

create_todos(["Buy groceries", "Finish extra lab", "Eat banana"])

Todo #1: Buy groceries
Todo #2: Finish extra lab
Todo #3: Eat banana

'Todo #1: Buy groceries\nTodo #2: Finish extra lab\nTodo #3: Eat banana\n'

In [10]:
mark_complete(1, "bought")

bought

Todo #1: Buy groceries
Todo #2: Finish extra lab
Todo #3: Eat banana

'Todo #1: [green][strike]Buy groceries[/strike][/green]\nTodo #2: Finish extra lab\nTodo #3: Eat banana\n'

In [11]:
create_todos_json = {
    "name": "create_todos",
    "description": "Add new todos from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [12]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the todo at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the todo to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the todo in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [13]:
tools = [{"type": "function", "function": create_todos_json},
        {"type": "function", "function": mark_complete_json}]

In [14]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [15]:
def loop(messages):
    done = False
    while not done:
        response = openai.chat.completions.create(model="gpt-5.2", messages=messages, tools=tools, reasoning_effort="none")
        finish_reason = response.choices[0].finish_reason
        if finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    show(response.choices[0].message.content)

In [16]:
system_message = """
You are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn.
Now use the todo list tools, create a plan, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [17]:
todos, completed = [], []
loop(messages)

Todo #1: Interpret the problem and define variables/assumptions (distance between Boston and New York).
Todo #2: Estimate the Boston–New York distance if not provided.
Todo #3: Set up relative motion equations with departure time offset.
Todo #4: Solve for meeting time and compute clock time.
Todo #5: Sanity-check the result and present final answer clearly.

Let D be the distance between Boston and New York (miles). Train B leaves Boston at 2:00 pm at 60 mph toward New 
York. Train N leaves New York at 3:00 pm at 80 mph toward Boston. They move toward each other along the same line.

Todo #1: Interpret the problem and define variables/assumptions (distance between Boston and New York).
Todo #2: Estimate the Boston–New York distance if not provided.
Todo #3: Set up relative motion equations with departure time offset.
Todo #4: Solve for meeting time and compute clock time.
Todo #5: Sanity-check the result and present final answer clearly.

Distance isn’t given, so use a reasonable common estimate: Boston–New York is about 215 miles (driving distance is 
~215–220; rail is similar). Take D = 215 miles.

Todo #1: Interpret the problem and define variables/assumptions (distance between Boston and New York).
Todo #2: Estimate the Boston–New York distance if not provided.
Todo #3: Set up relative motion equations with departure time offset.
Todo #4: Solve for meeting time and compute clock time.
Todo #5: Sanity-check the result and present final answer clearly.

...

Time after 3:00 pm to meet: t = (D − 60)/140 = (215 − 60)/140 = 155/140 ≈ 1.1071 hours ≈ 1 hour 6.4 minutes. 
Meeting time ≈ 3:00 pm + 1:06 ≈ 4:06 pm (about 4:06:26 pm).

Todo #1: Interpret the problem and define variables/assumptions (distance between Boston and New York).
Todo #2: Estimate the Boston–New York distance if not provided.
Todo #3: Set up relative motion equations with departure time offset.
Todo #4: Solve for meeting time and compute clock time.
Todo #5: Sanity-check the result and present final answer clearly.

Check: By ~4:06 pm, Boston train has traveled 2.1071 h × 60 ≈ 126.4 miles; NY train has traveled 1.1071 h × 80 ≈ 
88.6 miles; sum ≈ 215 miles = D. Consistent.

Todo #1: Interpret the problem and define variables/assumptions (distance between Boston and New York).
Todo #2: Estimate the Boston–New York distance if not provided.
Todo #3: Set up relative motion equations with departure time offset.
Todo #4: Solve for meeting time and compute clock time.
Todo #5: Sanity-check the result and present final answer clearly.

Assuming a typical Boston–New York distance of about 215 miles:

• By 3:00 pm, the Boston train has a 1-hour head start and has gone 60 miles, leaving 215 − 60 = 155 miles between 
them.  
• From 3:00 pm onward, they approach each other at 60 + 80 = 140 mph.  
• Time to meet after 3:00 pm:  
  t = 155/140 ≈ 1.107 hours ≈ 1 hour 6.4 minutes

So they meet at about 4:06 pm (approximately 4:06:26 pm).

In [27]:
messages

[{'role': 'system',
  'content': "\nYou are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn.\nNow use the todo list tools, create a plan, carry out the steps, and reply with the solution.\nIf any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.\nProvide your solution in Rich console markup without code blocks.\nDo not ask the user questions or clarification; respond only with the answer after using your tools.\n"},
 {'role': 'user',
  'content': '"\nA train leaves Boston at 2:00 pm traveling 60 mph.\nAnother train leaves New York at 3:00 pm traveling 80 mph toward Boston.\nWhen do they meet?\n'},
 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_LKD2aHD5Ou0xKePInxPOJ6tr', function=Function(arguments='{"descriptions":["Interpret the problem and def

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Now try to build an Agent Loop from scratch yourself!<br/>
            Create a new .ipynb and make one from first principles, referring back to this as needed.<br/>
            It's one of the few times that I recommend typing from scratch - it's a very satisfying result.
            </span>
        </td>
    </tr>
</table>